# Omni ASR 300M on `data_curated_levant_binary_v1` (exclude `qasr`)

This notebook builds a real OmniLingual mixture-parquet recipe dataset from:

- `/home/MohammadNabulsi/whisper/data_curated_levant_binary_v1`
- includes `casa`, `layla`, `masc`, and `omni`
- excludes every `qasr` shard

Then it launches the local OmniLingual 300M v2 training recipe with live log streaming.

The defaults below are sized for a single 80GB GPU, but you can still change the knobs in the settings cell before you run.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
from datetime import datetime

cwd = Path.cwd().resolve()
candidate_roots = [cwd, cwd / "asr_milestone7", cwd.parent]
project_root = None
for candidate in candidate_roots:
    if (candidate / "scripts").exists() and (candidate / "notebooks").exists() and (candidate / "README_ASR_PIPELINE.md").exists():
        project_root = candidate
        break

if project_root is None:
    raise RuntimeError(f"Could not locate asr_milestone7 root from cwd={cwd}")

repo_root = project_root.parent
dataset_root = repo_root / "data_curated_levant_binary_v1"
omni_repo = repo_root / ".omni_lingual_guide" / "omnilingual-asr"
python_bin = repo_root / ".venv" / "bin" / "python"
prepare_script = project_root / "scripts" / "prepare_omni_300m_levant_binary_no_qasr.py"

if not dataset_root.exists():
    raise FileNotFoundError(dataset_root)
if not omni_repo.exists():
    raise FileNotFoundError(omni_repo)
if not python_bin.exists():
    raise FileNotFoundError(python_bin)
if not prepare_script.exists():
    raise FileNotFoundError(prepare_script)

print(f"project_root={project_root}")
print(f"repo_root={repo_root}")
print(f"dataset_root={dataset_root}")
print(f"omni_repo={omni_repo}")
print(f"python_bin={python_bin}")


In [ ]:
# Main run settings
RUN_NAME = "omni_300m_levant_binary_no_qasr"
RUN_ROOT = project_root / "runs" / RUN_NAME
DATASET_CARD_NAME = "levant_binary_no_qasr_omni_300m_dataset"
MODEL_NAME = "omniASR_LLM_300M_v2"
TOKENIZER_NAME = "omniASR_tokenizer_written_v2"

# Throughput / memory knobs for a single 80GB GPU
TRAIN_STEPS = 5000
MAX_AUDIO_SECONDS = 75.0
MIN_AUDIO_SECONDS = 0.05
BATCH_SIZE = 8
MAX_NUM_ELEMENTS = 9_600_000
NUM_SEQS_MULTIPLE_OF = 1
GRAD_ACCUM = 1
LEARNING_RATE = 5e-5
VALIDATE_EVERY = 250
CHECKPOINT_EVERY = 250
PUBLISH_METRICS_EVERY = 25
BETA_CORPUS = 0.5
BETA_LANGUAGE = 0.5
EXAMPLE_SHUFFLE_WINDOW = 10000
BATCH_SHUFFLE_WINDOW = 64

# Control behavior
FORCE_REBUILD_DATASET = False
MAX_ROWS_PER_SPLIT = None  # set e.g. 100 for a tiny dry run

LOG_DIR = RUN_ROOT / "logs"
CHECKPOINT_ROOT = RUN_ROOT / "checkpoints"
PREP_LOG = LOG_DIR / "prepare.log"
TRAIN_LOG = LOG_DIR / "train.log"

print(f"run_root={RUN_ROOT}")
print(f"checkpoint_root={CHECKPOINT_ROOT}")
print(f"train_log={TRAIN_LOG}")


In [ ]:
def run_and_stream(cmd, *, cwd=None, env=None, log_path=None):
    cwd = str(cwd) if cwd is not None else None
    merged_env = os.environ.copy()
    if env:
        merged_env.update({k: str(v) for k, v in env.items()})
    print("Running:", " ".join(str(x) for x in cmd))
    if cwd:
        print("cwd:", cwd)
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=cwd,
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = []
    log_handle = log_path.open("w", encoding="utf-8") if log_path is not None else None
    try:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
            if log_handle is not None:
                log_handle.write(line)
                log_handle.flush()
    finally:
        if log_handle is not None:
            log_handle.close()
    return_code = proc.wait()
    if return_code != 0:
        cmd_text = " ".join(str(x) for x in cmd)
        raise RuntimeError(f"Command failed with exit code {return_code}: {cmd_text}")

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


In [ ]:
prepare_cmd = [
    python_bin,
    prepare_script,
    "--dataset-root", dataset_root,
    "--run-root", RUN_ROOT,
    "--omni-repo", omni_repo,
    "--dataset-card-name", DATASET_CARD_NAME,
    "--config-name", RUN_NAME,
    "--model-name", MODEL_NAME,
    "--tokenizer-name", TOKENIZER_NAME,
    "--train-steps", str(TRAIN_STEPS),
    "--max-audio-seconds", str(MAX_AUDIO_SECONDS),
    "--min-audio-seconds", str(MIN_AUDIO_SECONDS),
    "--batch-size", str(BATCH_SIZE),
    "--max-num-elements", str(MAX_NUM_ELEMENTS),
    "--num-seqs-multiple-of", str(NUM_SEQS_MULTIPLE_OF),
    "--grad-accumulation", str(GRAD_ACCUM),
    "--learning-rate", str(LEARNING_RATE),
    "--validate-every", str(VALIDATE_EVERY),
    "--checkpoint-every", str(CHECKPOINT_EVERY),
    "--publish-metrics-every", str(PUBLISH_METRICS_EVERY),
    "--beta-corpus", str(BETA_CORPUS),
    "--beta-language", str(BETA_LANGUAGE),
    "--example-shuffle-window", str(EXAMPLE_SHUFFLE_WINDOW),
    "--batch-shuffle-window", str(BATCH_SHUFFLE_WINDOW),
]
if FORCE_REBUILD_DATASET:
    prepare_cmd.append("--force-rebuild")
if MAX_ROWS_PER_SPLIT is not None:
    prepare_cmd.extend(["--max-rows-per-split", str(MAX_ROWS_PER_SPLIT)])

prepare_result = run_and_stream(prepare_cmd, cwd=project_root, log_path=PREP_LOG)
prepare_summary_path = RUN_ROOT / "prepare_summary.json"
prepare_summary = load_json(prepare_summary_path)

print("\nPrepared dataset summary path:", prepare_summary_path)
print(json.dumps({
    "counts_by_split": prepare_summary["counts_by_split"],
    "counts_by_corpus": prepare_summary["counts_by_corpus"],
    "counts_by_language": prepare_summary["counts_by_language"],
    "dataset_dir": prepare_summary["dataset_dir"],
    "config_path": prepare_summary["config_path"],
    "dataset_card_path": prepare_summary["dataset_card_path"],
    "skipped_rebuild": prepare_summary.get("skipped_rebuild", False),
}, ensure_ascii=False, indent=2))


In [ ]:
generated_config_path = Path(prepare_summary["config_path"])
print(generated_config_path.read_text(encoding="utf-8"))


In [ ]:
train_cmd = [
    python_bin,
    "-m",
    "workflows.recipes.wav2vec2.asr",
    CHECKPOINT_ROOT,
    "--config-file",
    generated_config_path,
]

train_env = {
    "PYTHONPATH": str(omni_repo),
    "PYTHONUNBUFFERED": "1",
    "HF_HOME": str(RUN_ROOT / "hf_cache"),
}

train_result = run_and_stream(train_cmd, cwd=omni_repo, env=train_env, log_path=TRAIN_LOG)
print("\nTraining log saved to:", TRAIN_LOG)


In [ ]:
checkpoint_dirs = sorted(CHECKPOINT_ROOT.glob("**/checkpoints/step_*/model"))
if checkpoint_dirs:
    latest_checkpoint = max(checkpoint_dirs, key=lambda p: p.stat().st_mtime_ns)
    print("Latest checkpoint:", latest_checkpoint)
else:
    print("No checkpoint model directories found yet under", CHECKPOINT_ROOT)

print("\nLast 100 lines of the training log:")
if TRAIN_LOG.exists():
    lines = TRAIN_LOG.read_text(encoding="utf-8", errors="replace").splitlines()
    for line in lines[-100:]:
        print(line)
else:
    print("missing log file")
